# BlindSpot v0.2: audit, evaluate, review

CPU-only follow-up: whole-possession development split, fixed stoppage proposal, error analysis, label-independent query benchmark and hash-verified stage checkpoints. Public footage has already been examined; these results are development evidence, not an independent test. Video and broadcast are optional and are not downloaded by default.


In [ ]:
%pip -q install numpy==2.2.6 pandas==2.2.3 gdown==5.2.0


In [ ]:
from pathlib import Path
import hashlib, json, os, zipfile, subprocess, sys
import gdown

PROJECT = Path('/content/blindspot-v02')
PROJECT.mkdir(exist_ok=True)
BUNDLE_ID = '1S5egsUB1Aybl22Ya4k8aHTYRBapgbSe0'
BUNDLE_SHA256 = 'bffffe793ec3b4a609fd0cc4c08ec6fafbeaa06b076e5dfdf14325125c636216'
def sha256(path):
    value = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()
package = Path('/content/BlindSpot_v02_project.zip')
if not package.exists() or sha256(package) != BUNDLE_SHA256:
    result = gdown.download(id=BUNDLE_ID, output=str(package), quiet=False)
    if result is None:
        raise RuntimeError('Bundle download failed; check sharing or Drive bandwidth limits')
assert sha256(package) == BUNDLE_SHA256, 'Project bundle checksum mismatch'
with zipfile.ZipFile(package) as archive:
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe bundle path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
print('Verified source bundle:', PROJECT)


In [ ]:
ASSETS = {'BAR-ATH_tactical_atd.txt': '1JEPmnLYax4dnVh1ZlInJAH5calmS1tIz', 'BAR-ATH_tactical_atd.xml': '1WBfy2sOwmJ2cX2w9XJq6y4VXo_kxKHjX', 'BAR-ATH_tactical_homography.csv': '1tuq2EGFmfvnXl8u5dU7JLoN3eNbrWwDJ', 'BAR-ATH_tactical_smart_tagging.xml': '1SvfntIRtFh6i7KAK1gVJk1NQbS_lus2K'}
EXPECTED = {'BAR-ATH_tactical_atd.txt': 'f130e350d2bfdf734b7e32f072931fccc6d015c1e837d082064875fe0a700431', 'BAR-ATH_tactical_atd.xml': '327e35e32e46a647389b88f883d4f9a2ab931e3e6148aef2fcf8f22b1e93c43a', 'BAR-ATH_tactical_homography.csv': '421c81abe16f0912dc45c2541f9bf19dd5660066397a3dcc043038a1b61c28c3', 'BAR-ATH_tactical_smart_tagging.xml': '0594d35e3df01b9c5002372d8615e2e8ef5379ac8f4e84c5ef7a53f4304c9019'}
data_dir = PROJECT / 'data/tactical'
data_dir.mkdir(parents=True, exist_ok=True)
for name, file_id in ASSETS.items():
    path = data_dir / name
    if not path.exists() or sha256(path) != EXPECTED[name]:
        result = gdown.download(id=file_id, output=str(path), quiet=False)
        if result is None:
            raise RuntimeError('Download failed: ' + name)
    assert sha256(path) == EXPECTED[name], 'Dataset checksum mismatch: ' + name
    print('Verified', name, path.stat().st_size, 'bytes')


## Optional durable checkpoints
Default: temporary runtime storage, with result ZIP download at the end. To preserve stages across a runtime reset, set PERSIST_TO_DRIVE=True before running this cell and authorize your own Google Drive mount. This keeps generated results in your own Drive, without changing sharing. Downloaded source/inputs remain checksum-verified.


In [ ]:
PERSIST_TO_DRIVE = False
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/BlindSpot/checkpoints/v02')
else:
    OUT = PROJECT / 'data/experiments/blindspot-v2'
OUT.mkdir(parents=True, exist_ok=True)
print('Checkpoint destination:', OUT, '| durable:', PERSIST_TO_DRIVE)


## Mathematical and data invariants

Tests check exact decoding against exhaustive paths, posterior marginals, reproducibility, slot-order invariance, team symmetry, halftime resets, and separation of training labels.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Run the follow-up

Complete H/A/D runs stay together. A minimum 250-frame embargo is rounded out to group boundaries. Previously observed public scores are not a fresh holdout. The fixed stoppage candidate stays separate. Completed stages resume only when source/config/input provenance and artifact hashes match.


In [ ]:
subprocess.run([sys.executable, 'scripts/run_followup.py', '--out', str(OUT), '--resume', '--archive'], check=True)


In [ ]:
subprocess.run([sys.executable, 'evaluation/validate_submission.py', str(OUT / 'predictions/evidence_v1.csv')], check=True)
results = json.loads((OUT / 'validation/scores.json').read_text())
import pandas as pd
display(pd.DataFrame(results['grouped_development']).T[['macro_f1','spell_f1_mean','scope']])
print(json.dumps(json.loads((OUT / 'validation/split.json').read_text()), indent=2))
print(json.dumps(json.loads((OUT / 'query_evaluation/query_benchmark.json').read_text())['random_30_seeds'], indent=2))
print(json.dumps(json.loads((OUT / 'predictions/checkpoint.json').read_text())['artifacts'], indent=2))


## Inspect development findings

The dashboard reports grouped development scores, stoppage errors and query error enrichment. Error discovery is not measured correction or reviewer time saved. The local video review lab supports synchronized clips; this default cloud run has no video.


In [ ]:
from IPython.display import HTML, display
display(HTML((OUT / 'index.html').read_text()))


## Save results before disconnecting

The archive contains the CSV, trained calibration weights, run manifest, aggregate scores, gap histories, boundary receipts, query controls, and interactive report. Keep match-derived diagnostics within the permitted sharing scope.

In [ ]:
import shutil
from google.colab import files
result_zip = shutil.make_archive('/content/BlindSpot_v02_results', 'zip', OUT)
files.download(result_zip)
